In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
titles = pd.read_excel("emp_titles.xlsx")

In [3]:
titles["emp_title"] = (titles["emp_title"]
                       .str.lower()
                       .str.replace("(","").str.replace(")","")
                       .str.replace(" 0 ","").str.replace(" 1 ","")
                       .str.replace(" 2 ","").str.replace(" 3 ","")
                       .str.replace(" 4 ","").str.replace(" 5 ","")
                       .str.replace(" 6 ","").str.replace(" 7 ","")
                       .str.replace(" 8 ","").str.replace(" 9 ","")
                       .str.replace('"',"").str.replace("'","")
                       .str.replace("V","").str.replace("I","")
                       .str.replace("lll","").str.replace("iii","")
                       .str.replace(" iv ","").str.replace("â€‹","")
                       .str.strip())

In [4]:
titles.value_counts()

emp_title              
teacher                    10241
manager                     9325
owner                       6190
registered nurse            4660
supervisor                  4515
                           ...  
fte, specialist                1
fte analyst                    1
ftc                            1
ftbs direct worker             1
zsenior project manager        1
Name: count, Length: 95683, dtype: int64

In [158]:
freq_titles = pd.read_csv("emp_words.tsv",sep="\t")

In [159]:
freq_titles.columns = ["word", "count"]

In [160]:
freq_titles.loc[freq_titles.word == "nurse","count"] += freq_titles[freq_titles.word == "rn"]["count"].item()
freq_titles.loc[freq_titles.word == "rn","count"] = 0

freq_titles.loc[freq_titles.word == "technician","count"] += (
    freq_titles[freq_titles.word == "tech"]["count"].item()
    +
    freq_titles[freq_titles.word == "technical"]["count"].item()
    )
freq_titles.loc[freq_titles.word == "tech","count"] = 0
freq_titles.loc[freq_titles.word == "technical","count"] = 0

freq_titles.loc[freq_titles.word == "administrator","count"] += (
    freq_titles[freq_titles.word == "admin"]["count"].item()
    +
    freq_titles[freq_titles.word == "administrative"]["count"].item()
    )
freq_titles.loc[freq_titles.word == "admin","count"] = 0
freq_titles.loc[freq_titles.word == "administrative","count"] = 0

freq_titles.loc[freq_titles.word == "account","count"] += (
    freq_titles[freq_titles.word == "accountant"]["count"].item()
    +
    freq_titles[freq_titles.word == "accounting"]["count"].item()
    )
freq_titles.loc[freq_titles.word == "accountant","count"] = 0
freq_titles.loc[freq_titles.word == "accounting","count"] = 0

freq_titles.loc[freq_titles.word == "service","count"] += freq_titles[freq_titles.word == "services"]["count"].item()
freq_titles.loc[freq_titles.word == "services","count"] = 0

freq_titles.loc[freq_titles.word == "manager","count"] += freq_titles[freq_titles.word == "mgr"]["count"].item()
freq_titles.loc[freq_titles.word == "mgr","count"] = 0


mgr - manager
vp - vice president
pr - Practice

In [161]:
freq_titles.nlargest(40,"count")

,word,count
13969,manager,64151
23748,technician,16781
7559,director,15827
20718,sales,14883
2521,assistant,14874
15818,nurse,13523
23495,teacher,11881
22976,supervisor,11850
22122,specialist,11607
8903,engineer,11596


In [131]:
freq_titles = freq_titles[~freq_titles["word"].isin(["of","operations","senior",
                                                     "project",'it','sr.','lead',
                                                     'registered','office','vice',
                                                     'support','general','customer',
                                                     'program','and','financial','human',
                                                     'sr','truck','software','hr',
                                                     'production','medical','management',
                                                     'rep','development','quality','store','&',
                                                     'vp','ii','field','team','regional',
                                                     'clinical','branch','finance','warehouse',
                                                     'network','care','loan','legal'])]

In [133]:
words_dictionary = dict(sorted(freq_titles.set_index("word").nlargest(50,"count").squeeze().to_dict().items(), key=lambda x:x[1], reverse=True))

In [138]:
count = 0
for x in words_dictionary.keys():
    words_dictionary[x] = count
    count+=1

In [139]:
words_dictionary

{'manager': 0,
 'technician': 1,
 'director': 2,
 'sales': 3,
 'assistant': 4,
 'nurse': 5,
 'teacher': 6,
 'supervisor': 7,
 'specialist': 8,
 'engineer': 9,
 'analyst': 10,
 'service': 11,
 'account': 12,
 'driver': 13,
 'officer': 14,
 'administrator': 15,
 'owner': 16,
 'coordinator': 17,
 'operator': 18,
 'associate': 19,
 'president': 20,
 'executive': 21,
 'business': 22,
 'consultant': 23,
 'clerk': 24,
 'maintenance': 25,
 'systems': 26,
 'marketing': 27,
 'mechanic': 28,
 'representative': 29,
 'security': 30,
 'agent': 31,
 'police': 32,
 'attorney': 33,
 'worker': 34,
 'therapist': 35,
 'staff': 36,
 'chief': 37,
 'principal': 38,
 'advisor': 39,
 'leader': 40,
 'foreman': 41,
 'developer': 42,
 'designer': 43,
 'secretary': 44,
 'professor': 45,
 'controller': 46,
 'server': 47,
 'banker': 48,
 'deputy': 49}

In [140]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Your list of employee titles
employee_titles = titles["emp_title"].to_list()

In [145]:
# calculate the most frequent features first
vect = TfidfVectorizer(vocabulary=words_dictionary, max_features=100)
matrix = vect.fit_transform(employee_titles)
top_features = vect.vocabulary_
n = len(top_features)

for feature, index in words_dictionary.items():
    top_features[feature] = index

# re-vectorize using both sets of features
# at this point len(top_features) == 13
vect = TfidfVectorizer(vocabulary=top_features)
X = vect.fit_transform(employee_titles)

In [155]:
# OPTIONAL
# Transform titles into TF-IDF features
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(employee_titles)

In [146]:
len(employee_titles)

395318

In [148]:
# Apply KMeans clustering
kmeans = KMeans(n_clusters=38, random_state=42, n_init = 'auto')
kmeans.fit(X)

# Get cluster labels
labels = kmeans.labels_

# Create clusters dictionary
clusters = {}
for i, title in enumerate(employee_titles):
    cluster_label = labels[i]
    if cluster_label not in clusters:
        clusters[cluster_label] = []
    clusters[cluster_label].append(title)

# Print the clusters
for cluster_label, cluster_titles in dict(sorted(clusters.items())).items():
    print(f"Cluster {cluster_label}:")
    print(cluster_titles[0:20])
    print("\n")

Cluster 0:
['day care proivder', 'letter carrier', 'carpenter', 'lead caterer', 'adv mtr proj fld rep', 'ac tech', 'admin asst', 'advance cnn', 'aftermarket expeditor', 'apprentice', 'architect', 'area leader', 'assembler', 'assembler', 'auditor', 'avp, private client support', 'baker', 'banker', 'barber', 'bi reports developer']


Cluster 1:
['assistant store manager', 'administration assistant', 'assistant branch manager', 'assistant butler', 'assistant manager', 'assistant principal', 'assistant store manager', 'assistant store manager in training', 'lab assistant', 'medical assistant', 'nursing assistant', 'ocuppational therapy assistant', 'school age care assistant', '!st assistant plumbing manager', '1950 assistant purchaser', '1st assistant', '1st assistant accountant', '1st assistant camera', '1st assistant d a', '1st assistant director']


Cluster 2:
['electronics manager', 'acc manager', 'accounting  manager', 'collection manager', 'cyber project manager', 'department manager

In [149]:
for cluster_label, cluster_titles in dict(sorted(clusters.items())).items():
    print(f"Cluster {cluster_label}:")
    print(len(cluster_titles))

Cluster 0:
134512
Cluster 1:
11829
Cluster 2:
51129
Cluster 3:
10816
Cluster 4:
7372
Cluster 5:
10030
Cluster 6:
13072
Cluster 7:
26
Cluster 8:
11110
Cluster 9:
12154
Cluster 10:
11312
Cluster 11:
11106
Cluster 12:
7881
Cluster 13:
8889
Cluster 14:
6864
Cluster 15:
3588
Cluster 16:
23
Cluster 17:
9289
Cluster 18:
6132
Cluster 19:
6508
Cluster 20:
4120
Cluster 21:
2163
Cluster 22:
3931
Cluster 23:
4424
Cluster 24:
5132
Cluster 25:
2259
Cluster 26:
4669
Cluster 27:
2515
Cluster 28:
5625
Cluster 29:
6433
Cluster 30:
2354
Cluster 31:
2661
Cluster 32:
2158
Cluster 33:
2680
Cluster 34:
2487
Cluster 35:
2743
Cluster 36:
3736
Cluster 37:
1586


In [157]:
print(clusters[13])

['licensed practical nurse', 'nurse practitioner', 'registered nurse', 'registered nurse', 'registered nurse', 'registered nurse', 'registered nurse', 'school nurse', 'access nurse', 'accreditation specialist - nurse lead', 'acting nurse manager', 'acute care nurse manager', 'acute care nurse practitioner', 'administrative  nurse', 'administrative nurse', 'administrative public health nurse', 'administrative registered nurse', 'admission nurse', 'admission nurse', 'admission nurse', 'admission nurse', 'admission nurse', 'admissions nurse', 'admissions nurse', 'admissions registered nurse', 'adult nurse practitioner', 'advance practice registered nurse', 'advance registered nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced practice nurse', 'advanced pra

In [192]:
# print(clusters[2])  # other
print(clusters[16]) # business / banker

# print(clusters[0])  # housekeeping
# print(clusters[1])  # parts 
# print(clusters[3])  # service 
# print(clusters[4])  # teacher
# print(clusters[6])  # owner
# print(clusters[7])  # assistant 
# print(clusters[8])  # engineer 
# print(clusters[10]) # consultant
# print(clusters[11]) # secretary
# print(clusters[12]) # analyst
# print(clusters[13]) # driver
# print(clusters[20]) # clerk
# print(clusters[21]) # officer
# print(clusters[23]) # technician
# print(clusters[24]) # specialist
# print(clusters[25]) # team lead 
# print(clusters[27]) # operator
# print(clusters[28]) # president
# print(clusters[29]) # administrator
# print(clusters[30]) # director
# print(clusters[31]) # associate
# print(clusters[33]) # payroll 
# print(clusters[34]) # office 
# print(clusters[36]) # professor
# print(clusters[37]) # attorney
# print(clusters[17]) # coordinator
# print(clusters[18]) # supervisor

# print(clusters[14]) # project manager
# print(clusters[5])  # general manager
# print(clusters[19]) # manager

# print(clusters[15]) # nurse
# print(clusters[32]) # rn

# print(clusters[9])  # marketing 
# print(clusters[22]) # sales

# print(clusters[26]) # executive BUT account
# print(clusters[35]) # accounting

['banker', 'vp business development', 'about business controls manager', 'accountant/business manager', 'ach business analysis manager', 'admin assistant/business manager', 'admissions / business office manager', 'ag and business banker', 'aml business liaison', 'application business engineer', 'area business leader', 'area business manager', 'area business manager', 'area business manager', 'area business manager', 'area business manager', 'area business manager', 'area business manager', 'area business unit manager', 'assistant business counselor', 'assistant business manager', 'assistant business manager', 'assistant business manager', 'assistant business manager', 'assistant business manager', 'assistant business manager', 'assistant business manager', 'assistant business manager', 'assistant business manager', 'assistant director, business development', 'assistant superintendent of business ser', 'assistant superintendent, business svc', 'associte business manager', 'asst director

In [195]:
clusters[16]

['banker',
 'vp business development',
 'about business controls manager',
 'accountant/business manager',
 'ach business analysis manager',
 'admin assistant/business manager',
 'admissions / business office manager',
 'ag and business banker',
 'aml business liaison',
 'application business engineer',
 'area business leader',
 'area business manager',
 'area business manager',
 'area business manager',
 'area business manager',
 'area business manager',
 'area business manager',
 'area business manager',
 'area business unit manager',
 'assistant business counselor',
 'assistant business manager',
 'assistant business manager',
 'assistant business manager',
 'assistant business manager',
 'assistant business manager',
 'assistant business manager',
 'assistant business manager',
 'assistant business manager',
 'assistant business manager',
 'assistant director, business development',
 'assistant superintendent of business ser',
 'assistant superintendent, business svc',
 'associte b

In [176]:
clusters_back = clusters.copy()

In [196]:
# print(clusters[16]) # business / banker

clusters_back[16] = [x for x in clusters[16] if ("bank" not in x)] # business 
clusters_back[39] = [x for x in clusters[16] if ("bank" in x)] # banker

In [177]:
# print(clusters[14]) # project manager
# print(clusters[5])  # general manager
# print(clusters[19]) # manager

managers = clusters_back[14] + clusters_back[5] + clusters_back[19]

clusters_back[5] = managers

clusters_back[14] = []

clusters_back[19] = []

In [178]:
# print(clusters[15]) # nurse
# print(clusters[32]) # rn

nurses = clusters_back[15] + clusters_back[32] 

clusters_back[15] = nurses

clusters_back[32] = []

In [179]:
# print(clusters[9])  # marketing 
# print(clusters[22]) # sales

sales_market = clusters_back[9] + clusters_back[22] 

clusters_back[9] = sales_market

clusters_back[22] = []

In [180]:
# print(clusters[26]) # executive BUT account
# print(clusters[35]) # accounting

accounts = [x for x in clusters_back[26] if ("account" in x)]

clusters_back[35] = accounts + clusters_back[35]

clusters_back[26] = [x for x in clusters_back[26] if ("account" not in x)]

In [167]:
for cluster_label, cluster_titles in dict(sorted(clusters_back.items())).items():
    print(f"Cluster {cluster_label}:")
    print(len(cluster_titles))

Cluster 0:
287
Cluster 1:
701
Cluster 2:
156175
Cluster 3:
6450
Cluster 4:
10820
Cluster 5:
36404
Cluster 6:
7193
Cluster 7:
11121
Cluster 8:
11270
Cluster 9:
13414
Cluster 10:
4551
Cluster 11:
1615
Cluster 12:
11137
Cluster 13:
8610
Cluster 14:
1
Cluster 15:
11985
Cluster 16:
3743
Cluster 17:
6437
Cluster 18:
9890
Cluster 19:
1
Cluster 20:
3363
Cluster 21:
8447
Cluster 22:
1
Cluster 23:
7934
Cluster 24:
10992
Cluster 25:
2643
Cluster 26:
2780
Cluster 27:
5968
Cluster 28:
5574
Cluster 29:
3802
Cluster 30:
12702
Cluster 31:
4962
Cluster 32:
1
Cluster 33:
810
Cluster 34:
4302
Cluster 35:
5487
Cluster 36:
1549
Cluster 37:
2200


In [205]:
# CHECK IF GOOD
summary = 0
for x in clusters_back.values():
    summary += len(x)

print(summary)

395318


In [182]:
new_employee_titles = clusters_back[2]

In [183]:
# Transform titles into TF-IDF features
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(new_employee_titles)

In [184]:
# Apply KMeans clustering
kmeans = KMeans(n_clusters=10, random_state=42, n_init = 'auto')
kmeans.fit(X)

# Get cluster labels
labels = kmeans.labels_

# Create clusters dictionary
new_clusters = {}
for i, title in enumerate(new_employee_titles):
    cluster_label = labels[i]
    if cluster_label not in new_clusters:
        new_clusters[cluster_label] = []
    new_clusters[cluster_label].append(title)

In [187]:
for cluster_label, cluster_titles in dict(sorted(new_clusters.items())).items():
    print(f"Cluster {cluster_label}:")
    print(cluster_titles[0:20])
    print()

Cluster 0:
['administrative.shipping mrg', 'const. mrg.', 'dept mrg', 'gernal mrg', 'logistics mrg', 'mrg documenation control']

Cluster 1:
['admin systems mgr', 'administrative systems mgmr', 'advisor enterprise systems', 'aircraft armament systems', 'aircraft fuel systems', 'airways transportation systems specialis', 'alignment systems manager', 'assistant manager, pharmacy systems supp', 'assoc. info. systems analystspecialist', 'associate information systems anyalst', 'asst direct project information systems', 'at water systems tech', 'av/video systems engineer', 'aviation transportation systems manager', 'avionics systems helper', 'avp clinical information systems', 'avp, systems admin', 'billing systems manager', 'billing systems manager', 'building systems mananger']

Cluster 2:
['membership representative', 'a&d representative', 'a/p representative', 'a/r field representative', 'a/r representative', 'access care representative', 'access point support representative', 'access r

In [189]:
for cluster_label, cluster_titles in dict(sorted(new_clusters.items())).items():
    print(f"Cluster {cluster_label}:")
    print(len(cluster_titles))

Cluster 0:
6
Cluster 1:
503
Cluster 2:
1793
Cluster 3:
5542
Cluster 4:
12081
Cluster 5:
1802
Cluster 6:
1596
Cluster 7:
877
Cluster 8:
128176
Cluster 9:
3799


In [144]:
# print(new_clusters[0]) # other
# print(new_clusters[1]) # systems
# print(new_clusters[2]) # representative
# print(new_clusters[3]) # tech
# print(new_clusters[4]) # manager
# print(new_clusters[5]) # maintenance 
# print(new_clusters[6]) # designer
# print(new_clusters[7]) # machinist
# print(new_clusters[8]) # other
# print(new_clusters[9]) # other

['founder & principal / senior researcher', 'senior ce', 'accountant senior', 'administrative senior pastor', 'administrative support senior', 'advisory senior manager', 'aircraft mechanic senior', 'applecare senior advisor', 'assembler senior', 'assistant vp, senior technology developr', 'assurance senior', 'assurance senior', 'assurance senior', 'assurance senior manager', 'assurance senior manager', 'assurance services senior', 'audio visual senior acct. manager', 'audit senior', 'audit senior', 'audit senior', 'audit senior', 'audit senior', 'audit senior', 'audit senior', 'audit senior', 'audit senior manager', 'audit senior manager', 'audit senior manager', 'avionics installer senior', 'avp -senior field examiner', 'avp senior underwritier', 'avp, senior auditor 1', 'avp; senior investigator', 'bank officer senior customer care spec.', 'battalion senior medic', 'bbt senior', 'bio med senior', 'bn senior medic', 'branch senior', 'casesmart senior flex developer', 'cc senior manage

In [190]:
clusters_back[5] = clusters_back[5] + new_clusters[4] # manager
clusters_back[23] = clusters_back[23] + new_clusters[3] # tech
clusters_back[14] = new_clusters[1] # systems
clusters_back[19] = new_clusters[2] # representative
clusters_back[22] = new_clusters[5] # maintenance
clusters_back[32] = new_clusters[6] # designer
clusters_back[38] = new_clusters[7] # machinist

clusters_back[2] = new_clusters[0] + new_clusters[8] + new_clusters[9] # other

In [197]:
for cluster_label, cluster_titles in dict(sorted(clusters_back.items())).items():
    print(f"Cluster {cluster_label}:")
    print(len(cluster_titles))

Cluster 0:
287
Cluster 1:
701
Cluster 2:
131981
Cluster 3:
6450
Cluster 4:
10820
Cluster 5:
48485
Cluster 6:
7193
Cluster 7:
11121
Cluster 8:
11270
Cluster 9:
13414
Cluster 10:
4551
Cluster 11:
1615
Cluster 12:
11137
Cluster 13:
8610
Cluster 14:
503
Cluster 15:
11985
Cluster 16:
2452
Cluster 17:
6437
Cluster 18:
9890
Cluster 19:
1793
Cluster 20:
3363
Cluster 21:
8447
Cluster 22:
1802
Cluster 23:
13476
Cluster 24:
10992
Cluster 25:
2643
Cluster 26:
2780
Cluster 27:
5968
Cluster 28:
5574
Cluster 29:
3802
Cluster 30:
12702
Cluster 31:
4962
Cluster 32:
1596
Cluster 33:
810
Cluster 34:
4302
Cluster 35:
5487
Cluster 36:
1549
Cluster 37:
2200
Cluster 38:
877
Cluster 39:
1291


In [198]:
final_clusters = dict()

In [ ]:
# print(clusters[2])  # other
# print(clusters[16]) # business / banker

# print(clusters[0])  # housekeeping
# print(clusters[1])  # parts 
# print(clusters[3])  # service 
# print(clusters[4])  # teacher
# print(clusters[6])  # owner
# print(clusters[7])  # assistant 
# print(clusters[8])  # engineer 
# print(clusters[10]) # consultant
# print(clusters[11]) # secretary
# print(clusters[12]) # analyst
# print(clusters[13]) # driver
# print(clusters[20]) # clerk
# print(clusters[21]) # officer
# print(clusters[23]) # technician
# print(clusters[24]) # specialist
# print(clusters[25]) # team lead 
# print(clusters[27]) # operator
# print(clusters[28]) # president
# print(clusters[29]) # administrator
# print(clusters[30]) # director
# print(clusters[31]) # associate
# print(clusters[33]) # payroll 
# print(clusters[34]) # office 
# print(clusters[36]) # professor
# print(clusters[37]) # attorney
# print(clusters[17]) # coordinator
# print(clusters[18]) # supervisor

# print(clusters[14]) # project manager
# print(clusters[5])  # general manager
# print(clusters[19]) # manager

# print(clusters[15]) # nurse
# print(clusters[32]) # rn

# print(clusters[9])  # marketing 
# print(clusters[22]) # sales

# print(clusters[26]) # executive BUT account
# print(clusters[35]) # accounting

In [242]:
print(clusters_back[39][0:20])

['banker', 'ag and business banker', 'avp personal business banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker', 'banker']


In [243]:
final_clusters["housekeeping"] = clusters_back[0]
final_clusters["parts"] = clusters_back[1]
final_clusters["other"] = clusters_back[2]
final_clusters["service"] = clusters_back[3]
final_clusters["teacher"] = clusters_back[4]
final_clusters["manager"] = clusters_back[5]
final_clusters["owner"] = clusters_back[6]
final_clusters["assistant"] = clusters_back[7]
final_clusters["engineer"] = clusters_back[8]
final_clusters["marketing and sales"] = clusters_back[9]
final_clusters["consultant"] = clusters_back[10]
final_clusters["secretary"] = clusters_back[11]
final_clusters["analyst"] = clusters_back[12]
final_clusters["driver"] = clusters_back[13]
final_clusters["systems "] = clusters_back[14]
final_clusters["nurse"] = clusters_back[15]
final_clusters["business"] = clusters_back[16]
final_clusters["coordinator"] = clusters_back[17]
final_clusters["supervisor"] = clusters_back[18]
final_clusters["representative"] = clusters_back[19]
final_clusters["clerk"] = clusters_back[20]
final_clusters["officer"] = clusters_back[21]
final_clusters["maintenance"] = clusters_back[22]
final_clusters["technician"] = clusters_back[23]
final_clusters["specialist"] = clusters_back[24]
final_clusters["leader"] = clusters_back[25]
final_clusters["executive"] = clusters_back[26]
final_clusters["operator"] = clusters_back[27]
final_clusters["president"] = clusters_back[28]
final_clusters["administrator"] = clusters_back[29]
final_clusters["director"] = clusters_back[30]
final_clusters["associate"] = clusters_back[31]
final_clusters["designer"] = clusters_back[32]
final_clusters["payroll"] = clusters_back[33]
final_clusters["office"] = clusters_back[34]
final_clusters["account"] = clusters_back[35]
final_clusters["professor"] = clusters_back[36]
final_clusters["attorney"] = clusters_back[37]
final_clusters["machinist"] = clusters_back[38]
final_clusters["banker"] = clusters_back[39]

In [244]:
# CHECK IF GOOD
summary = 0
for x in final_clusters.values():
    summary += len(x)

print(summary)

395318


In [251]:
final_clusters_df = pd.DataFrame.from_dict(final_clusters, orient='index').transpose()

In [252]:
final_clusters_df.to_excel("titles_clusters.xlsx")

---

In [2]:
import pandas as pd

In [5]:
employee_dictionary = pd.read_excel("titles_clusters_good.xlsx")

In [25]:
dataset = pd.read_excel("predicted_data_accepted_new.xlsx")

In [14]:
employee_dictionary = employee_dictionary.iloc[:,1:]

In [18]:
employee_dictionary.drop("other", axis = 1, inplace=True)

In [22]:
employee_dictionary.transpose()

,index,0,1,2,3,4,5,6,7,8,...,132088,132089,132090,132091,132092,132093,132094,132095,132096,132097
0,housekeeping,housekeeping,account manager for housekeeping,assistant director for housekeeping serv,assistant director housekeeping,assistant director of housekeeping,assistant director of housekeeping,assistant director of housekeeping,assistant director of housekeeping,assistant housekeeping manager,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,parts,parts manager,aftermarket parts project manager,air craft parts mechanic,aircraft parts clerk,aircraft parts repairer,assistant parts manager,assistant parts manager,assistant parts manager,assistant parts manager,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,service,customer service representative,dining service,public service supervisor,service deli,service provider,a/c service tech,access service rep,account service,account service manager,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,teacher,assistant teacher,teacher,teacher,teacher,teacher,teacher,teacher assistant,a teacher,a.teacher,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,manager,cyber project manager,project director,project manager,account/ project manager,account/project manager,administration/project manager,administrative project manager,agile project manager,agronomy project manager,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,owner,co-owner,`owner/mechanic,"a member, owner",accountant - owner,accountant/owner,advisor/owner,agent - owner,agent - owner,agent owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,assistant,assistant store manager,administration assistant,assistant branch manager,assistant butler,assistant manager,assistant principal,assistant store manager,assistant store manager in training,lab assistant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,engineer,building engineer,1st engineer,2 nd engineer,2nd assistant engineer,2nd engineer,2nd engineer,911 engineer,a/v engineer,access management engineer,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,marketing and sales,event marketing manager,marketing manager,regional marketing coordinator,academic marketing,accounts manager / marketing,activity/marketing director,ad marketing,ad marketing,ad sales marketing manager,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,consultant,consultant,consultant,leasing & sales consultant,401k education consultant,academic consultant,account consultant,account consultant,account consultant,account consultant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [82]:
def cleaning (x : str) -> str:
    if x:
        return (str(x).lower()
                    .replace("(","").replace(")","")
                    .replace(" 0 ","").replace(" 1 ","")
                    .replace(" 2 ","").replace(" 3 ","")
                    .replace(" 4 ","").replace(" 5 ","")
                    .replace(" 6 ","").replace(" 7 ","")
                    .replace(" 8 ","").replace(" 9 ","")
                    .replace('"',"").replace("'","")
                    .replace("V","").replace("I","")
                    .replace("lll","").replace("iii","")
                    .replace(" iv ","").replace("â€‹","")
                    .strip())
    return None

In [49]:
dataset.iloc[19,:]

Unnamed: 0                                19
funded_amnt                            10000
term                               36 months
grade                                      5
emp_title                Contract Specialist
emp_length                               4.0
home_ownership                      MORTGAGE
annual_inc                           91392.0
loan_status                       Fully Paid
purpose                                house
zip_code                               644xx
addr_state                          Missouri
dti                                     9.44
delinq_2yrs                                0
fico_range_low                           710
predicted_loan_status             Fully Paid
emp_cluster                              NaN
Name: 19, dtype: object

In [70]:
clear_clusters = dict()

In [71]:
for clust in employee_dictionary.columns:
    clear_clusters[clust] = employee_dictionary[clust].value_counts().index

In [88]:
for x,li in clear_clusters.items():
    print(x + ":")
    print(len(li))

housekeeping:
64
parts:
215
service:
1370
teacher:
75
manager:
8481
owner:
318
assistant:
2061
engineer:
2001
marketing and sales:
1696
consultant:
1267
secretary:
245
analyst:
2880
driver:
480
systems :
349
nurse:
472
business:
860
coordinator:
2728
supervisor:
1478
representative:
719
clerk:
1018
officer:
1493
maintenance:
431
technician:
4227
specialist:
4954
leader:
1096
executive:
308
operator:
1365
president:
813
administrator:
982
director:
4088
associate:
1618
designer:
427
payroll:
276
office:
412
account:
821
professor:
209
attorney:
189
machinist:
92
banker:
122


In [50]:
employee_dictionary.drop_duplicates()

,housekeeping,parts,service,teacher,manager,owner,assistant,engineer,marketing and sales,consultant,...,director,associate,designer,payroll,office,account,professor,attorney,machinist,banker
0,housekeeping,parts manager,customer service representative,assistant teacher,cyber project manager,co-owner,assistant store manager,building engineer,event marketing manager,consultant,...,director,3rd key sales associate,3d designer,payroll supervisor,office specialist,capital account manager,associate professor,administrative attorney,aircraft worker machinist,banker
1,account manager for housekeeping,aftermarket parts project manager,dining service,teacher,project director,`owner/mechanic,administration assistant,1st engineer,marketing manager,consultant,...,assitant director,401k plan sales associate,academic designer,a/p payroll supervisor,account manager/office manager,academic account manager,academic counselor / professor,agency attorney,areospace machinist,ag and business banker
2,assistant director for housekeeping serv,air craft parts mechanic,public service supervisor,teacher,project manager,"a member, owner",assistant branch manager,2 nd engineer,regional marketing coordinator,leasing & sales consultant,...,asst. director of support services,529 senior associate,academic designer,a/p supervisor/payroll admin,account/office manager,account,adj assistant professor,agency attorney,assembler / machinist,avp personal business banker
3,assistant director housekeeping,aircraft parts clerk,service deli,teacher,account/ project manager,accountant - owner,assistant butler,2nd assistant engineer,academic marketing,401k education consultant,...,asst. site director,529 senior associate,advanced designer,accounting & payroll clerk,accountant/office manager,account,adjuct professor,agency attorney,automotive machinist,banker
4,assistant director of housekeeping,aircraft parts repairer,service provider,teacher,account/project manager,accountant/owner,assistant manager,2nd engineer,accounts manager / marketing,academic consultant,...,director of institutional research,a/p associate,advanced senior designer,accounting payroll,accountant/office manager,account,adjunct assistant professor,agency attorney,aviation machinist,banker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132093,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [92]:
mapper = dict()

In [93]:
for k, v_li in clear_clusters.items():
    for v in v_li:
        mapper[v] = k 

In [96]:
dataset["emp_cluster"] = (
    dataset["emp_title"]
    .copy()
    .apply(lambda x: cleaning(x) if x else pd.NA)
    .map(mapper)
    .astype("str", errors = 'ignore')
)

In [97]:
dataset.isna().sum()

Unnamed: 0                   0
funded_amnt                  0
term                         0
grade                        0
emp_title                23833
emp_length               23779
home_ownership               0
annual_inc                   0
loan_status                  0
purpose                      0
zip_code                     0
addr_state                   0
dti                          0
delinq_2yrs                  0
fico_range_low               0
predicted_loan_status        0
emp_cluster                  0
dtype: int64

In [101]:
dataset = dataset.iloc[:,1:]

In [103]:
dataset["emp_cluster"].value_counts()

emp_cluster
nan                    155904
manager                 48520
technician              13473
marketing and sales     13413
director                12699
nurse                   11985
engineer                11268
analyst                 11137
assistant               11118
specialist              10991
teacher                 10818
supervisor               9890
driver                   8609
officer                  8447
owner                    7193
service                  6450
coordinator              6436
operator                 5968
president                5574
account                  5486
associate                4961
consultant               4551
office                   4298
administrator            3801
clerk                    3360
executive                2780
leader                   2487
business                 2452
attorney                 2200
maintenance              1802
representative           1792
secretary                1614
designer                 159

In [105]:
dataset.to_excel("predicted_data_accepted_new_full.xlsx")